# Preparation GTP

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
!git clone https://github.com/YmShan/SDTrack.git
!cd SDTrack/SDTrack-Event

import os
import shutil
import re
from pathlib import Path
from tqdm.notebook import tqdm
import time

In [ ]:
#Copy GTP script
shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/csv/GTP_FE108_csv.py',
    '/content/SDTrack/SDTrack-Event/GTP_FE108.py')

In [ ]:
initials = 'seya'
sequences = ['key_1:2hz', 'key_1hz', 'key_2hz', 'lamp_1:2hz', 'lamp_1hz', 'lamp_2hz',
             'repeater_1:2hz', 'repeater_1hz', 'repeater_2hz', 'soap_1:2hz',
             'soap_1hz', 'soap_2hz', 'white_1:2hz', 'white_1hz', 'white_2hz',
             'cleaner_1:2hz', 'cleaner_1hz', 'cleaner_2hz', 'knife_1:2hz', 'knife_1hz', 'knife_2hz',
             'longus_1:2hz', 'longus_1hz', 'longus_2hz', 'nonstop_1:2hz',
             'nonstop_1hz', 'nonstop_2hz', 'rexona_1:2hz', 'rexona_1hz', 'rexona_2hz']

In [ ]:
#Copy needed sequences

for seq in tqdm(sequences):

    os.makedirs(f'/content/SDTrack/SDTrack-Event/data/test/{seq}', exist_ok=True)
    os.makedirs('/content/SDTrack/SDTrack-Event/data/train/', exist_ok=True)


    shutil.copy(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/events.csv',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}/events.csv'
    )

    shutil.copytree(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/aps',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}/img'
    )

In [ ]:
# Create test.txt files

with open('/content/SDTrack/SDTrack-Event/data/test/test.txt', 'w') as f:
    for seq in sequences:
        f.write(f'{seq}\n')

#Create empty train/test.txt and
os.makedirs('/content/SDTrack/SDTrack-Event/data/train', exist_ok=True)
with open('/content/SDTrack/SDTrack-Event/data/train/test.txt', 'w') as f:
    f.write('')

In [ ]:
#Run script to generate GTP frames

!python /content/SDTrack/SDTrack-Event/GTP_FE108.py \
    --target_dir /content/SDTrack/SDTrack-Event/data \
    --stack_name inter1_stack_3008 \
    --T_num 1 \
    --stack_amount_1c2c 30 \
    --stack_amount_3c 30 \
    --decay_rate_3c 0.8

# Run sequences with flicker(before any changes and fine_tune)

In [ ]:
!pip install jpeg4py
!pip install lmdb
!pip install torchinfo
!pip install visdom
!pip install spikingjelly

In [ ]:
%cd /content/

!git clone https://github.com/YmShan/SDTrack.git
!cd SDTrack/SDTrack-Event

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

import os
import shutil
import re
from pathlib import Path
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

In [ ]:
#Put checkpoints into the correct folders

os.makedirs('/content/SDTrack/SDTrack-Event/pretrained_models', exist_ok=True)
os.makedirs('/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-fe108', exist_ok=True)

shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/SDTrack-tiny-1x4.pth',
    '/content/SDTrack/SDTrack-Event/pretrained_models/'
)

shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/SDTrack_ep0100.pth.tar',
    '/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-fe108/'
)


In [ ]:
initials = 'seya'
sequences = ['key_1:2hz', 'key_1hz', 'key_2hz', 'lamp_1:2hz', 'lamp_1hz', 'lamp_2hz',
             'repeater_1:2hz', 'repeater_1hz', 'repeater_2hz', 'soap_1:2hz',
             'soap_1hz', 'soap_2hz', 'white_1:2hz', 'white_1hz', 'white_2hz',
             'cleaner_1:2hz', 'cleaner_1hz', 'cleaner_2hz', 'knife_1:2hz', 'knife_1hz', 'knife_2hz',
             'longus_1:2hz', 'longus_1hz', 'longus_2hz', 'nonstop_1:2hz',
             'nonstop_1hz', 'nonstop_2hz', 'rexona_1:2hz', 'rexona_1hz', 'rexona_2hz']

In [ ]:
#Copy needed sequences

for seq in tqdm(sequences):

    os.makedirs(f'/content/SDTrack/SDTrack-Event/data/test/{seq}', exist_ok=True)
    os.makedirs('/content/SDTrack/SDTrack-Event/data/train/', exist_ok=True)

    shutil.copy(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/groundtruth_rect.txt',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}'
    )

    shutil.copytree(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/aps',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}/img'
    )

    shutil.copytree(
    f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/inter1_stack_3008',
    f'/content/SDTrack/SDTrack-Event/data/test/{seq}/inter1_stack_3008'
    )


In [ ]:
#Run SDTracker

%cd /content/SDTrack/SDTrack-Event
!python tracking/test.py SDTrack SDTrack-tiny-fe108 --dataset eotb --threads 4 --num_gpus 1


In [ ]:
#Get metrics (PR and SR)

def calc_metrics(gt, pr):
        min_len = min(len(gt), len(pr))
        gt = gt[:min_len]
        pr = pr[:min_len]

        gt_centers = gt[:, :2] + gt[:, 2:4] / 2
        pr_centers = pr[:, :2] + pr[:, 2:4] / 2
        distances  = np.sqrt(np.sum((gt_centers - pr_centers) ** 2, axis=1))
        precision  = np.mean(distances <= 20)

        ix1 = np.maximum(gt[:, 0], pr[:, 0])
        iy1 = np.maximum(gt[:, 1], pr[:, 1])
        ix2 = np.minimum(gt[:, 0] + gt[:, 2], pr[:, 0] + pr[:, 2])
        iy2 = np.minimum(gt[:, 1] + gt[:, 3], pr[:, 1] + pr[:, 3])

        inter_area = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
        union_area = gt[:, 2] * gt[:, 3] + pr[:, 2] * pr[:, 3] - inter_area
        iou        = inter_area / (union_area + 1e-10)
        success    = np.mean(iou > 0.5)

        return precision, success


rows = []
for seq in sequences:
    gt = f'/content/SDTrack/SDTrack-Event/data/test/{seq}/groundtruth_rect.txt'
    pr = f'/content/SDTrack/SDTrack-Event/output/test/tracking_results/SDTrack/SDTrack-tiny-fe108/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (SD)":     pr,
        "SR@0.5 (SD)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (SD)'].mean():.3f}   {df['SR@0.5 (SD)'].mean():.3f}")

# Create augmented sequences

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import pandas as pd
from tqdm.notebook import tqdm

import os
import plotly.graph_objects as go


In [ ]:
# Augmentation A

class FlashAugmentor:
    def __init__(self, csv_path, aps_dir, h=260, w=346):
        # load events
        df = pd.read_csv(csv_path)
        self.events = df[['t', 'x', 'y', 'p']].values.astype(np.int64)

        # build boundaries from aps frame count
        num_frames = len(sorted(os.listdir(aps_dir)))
        t_min = int(self.events[0, 0])
        t_max = int(self.events[-1, 0])
        self.boundaries = np.linspace(t_min, t_max, num_frames + 1, dtype=np.int64)
        self.num_frames = num_frames

        self.h = h
        self.w = w
        self.total_pixels = h * w

    def augment(self):
        flashes = self._plan_flashes()

        all_new_events = []
        for start_frame, duration in flashes:
            flash_events = self._generate_flash(start_frame, duration)
            all_new_events.append(flash_events)

        if all_new_events:
            all_new_events = np.concatenate(all_new_events, axis=0)
            result = np.concatenate([self.events, all_new_events], axis=0)
        else:
            result = self.events.copy()

        result = result[result[:, 0].argsort()]
        return result

    def _plan_flashes(self):
        flashes = []
        current_frame = np.random.randint(8, 46)
        while current_frame < self.num_frames:
            duration = np.random.choice([5, 6, 7])
            actual_duration = min(duration, self.num_frames - current_frame)
            flashes.append((current_frame, actual_duration))
            current_frame += actual_duration + np.random.randint(15, 41)
        return flashes

    def _generate_flash(self, start_frame, duration):
        all_events = []

        if duration >= 3:
            n_pos = np.random.randint(1, duration - 1)
            n_neg = duration - n_pos - 1
            n_transition = 1
        elif duration == 2:
            n_pos, n_transition, n_neg = 1, 0, 1
        else:
            n_pos, n_transition, n_neg = 1, 0, 0

        frame_idx = start_frame
        for _ in range(n_pos):
            t_start = int(self.boundaries[frame_idx])
            t_end = int(self.boundaries[frame_idx + 1])
            all_events.append(self._generate_frame_events(t_start, t_end, 'pos'))
            frame_idx += 1

        if n_transition == 1:
            t_start = int(self.boundaries[frame_idx])
            t_end = int(self.boundaries[frame_idx + 1])
            all_events.append(self._generate_frame_events(t_start, t_end, 'transition'))
            frame_idx += 1

        for _ in range(n_neg):
            t_start = int(self.boundaries[frame_idx])
            t_end = int(self.boundaries[frame_idx + 1])
            all_events.append(self._generate_frame_events(t_start, t_end, 'neg'))
            frame_idx += 1

        return np.concatenate(all_events, axis=0)

    def _generate_frame_events(self, t_start, t_end, phase):
        if phase == 'pos':
            active_ratio = np.random.uniform(0.50, 0.85)
            pos_ratio = np.random.uniform(0.95, 0.99)
        elif phase == 'transition':
            active_ratio = np.random.uniform(0.01, 0.05)
            pos_ratio = np.random.uniform(0.55, 0.65)
        else:  # neg
            active_ratio = np.random.uniform(0.85, 1.00)
            pos_ratio = np.random.uniform(0.01, 0.05)

        n_pixels = int(self.total_pixels * active_ratio)


        indices = np.random.choice(self.total_pixels, size=n_pixels, replace=False)
        xs = indices % self.w
        ys = indices // self.w

        result = []
        for x, y in zip(xs, ys):
            n_events = np.random.randint(1, 21)
            n_pos = int(round(n_events * pos_ratio))
            n_neg = n_events - n_pos

            timestamps = np.random.randint(t_start, t_end, size=n_events)
            polarities = np.array([1] * n_pos + [0] * n_neg)
            np.random.shuffle(polarities)

            for t, p in zip(timestamps, polarities):
                result.append([t, x, y, p])

        return np.array(result, dtype=np.int64)

In [ ]:
# Augmentation B

class FlashAugmentor2:
    def __init__(self, csv_path, aps_dir, h=260, w=346):
        # load events
        df = pd.read_csv(csv_path)
        self.events = df[['t', 'x', 'y', 'p']].values.astype(np.int64)

        # build boundaries from aps frame count
        num_frames = len(sorted(os.listdir(aps_dir)))
        t_min = int(self.events[0, 0])
        t_max = int(self.events[-1, 0])
        self.boundaries = np.linspace(t_min, t_max, num_frames + 1, dtype=np.int64)
        self.num_frames = num_frames

        self.h = h
        self.w = w
        self.total_pixels = h * w

    def augment(self):
        flashes = self._plan_flashes()

        all_new_events = []
        for start_frame, duration in flashes:
            flash_events = self._generate_flash(start_frame, duration)
            all_new_events.append(flash_events)

        if all_new_events:
            all_new_events = np.concatenate(all_new_events, axis=0)
            result = np.concatenate([self.events, all_new_events], axis=0)
        else:
            result = self.events.copy()

        result = result[result[:, 0].argsort()]
        return result, flashes

    def _plan_flashes(self):
        flashes = []
        current_frame = np.random.randint(1, 6)
        while current_frame < self.num_frames:
            duration = np.random.randint(7, 14)
            actual_duration = min(duration, self.num_frames - current_frame)
            flashes.append((current_frame, actual_duration))
            current_frame += actual_duration + np.random.randint(2, 12)
        return flashes

    def _generate_flash(self, start_frame, duration):
        all_events = []

        if duration >= 3:
            n_pos = np.random.randint(1, duration - 1)
            n_neg = duration - n_pos - 1
            n_transition = 1
        elif duration == 2:
            n_pos, n_transition, n_neg = 1, 0, 1
        else:
            n_pos, n_transition, n_neg = 1, 0, 0

        frame_idx = start_frame
        for _ in range(n_pos):
            t_start = int(self.boundaries[frame_idx])
            t_end = int(self.boundaries[frame_idx + 1])
            all_events.append(self._generate_frame_events(t_start, t_end, 'pos'))
            frame_idx += 1

        if n_transition == 1:
            t_start = int(self.boundaries[frame_idx])
            t_end = int(self.boundaries[frame_idx + 1])
            all_events.append(self._generate_frame_events(t_start, t_end, 'transition'))
            frame_idx += 1

        for _ in range(n_neg):
            t_start = int(self.boundaries[frame_idx])
            t_end = int(self.boundaries[frame_idx + 1])
            all_events.append(self._generate_frame_events(t_start, t_end, 'neg'))
            frame_idx += 1

        return np.concatenate(all_events, axis=0)

    def _generate_frame_events(self, t_start, t_end, phase):
        if phase == 'pos':
            active_ratio = np.random.uniform(0.50, 0.85)
            pos_ratio = np.random.uniform(0.95, 0.99)
        elif phase == 'transition':
            active_ratio = np.random.uniform(0.01, 0.05)
            pos_ratio = np.random.uniform(0.55, 0.65)
        else:  # neg
            active_ratio = np.random.uniform(0.85, 1.00)
            pos_ratio = np.random.uniform(0.01, 0.05)

        n_pixels = int(self.total_pixels * active_ratio)
        indices = np.random.choice(self.total_pixels, size=n_pixels, replace=False)
        xs = indices % self.w
        ys = indices // self.w

        n_events_per_pixel = np.random.randint(1, 11, size=n_pixels)
        total_events = n_events_per_pixel.sum()

        timestamps = np.random.randint(t_start, t_end, size=total_events)
        polarities = (np.random.rand(total_events) < pos_ratio).astype(np.int64)

        xs_repeated = np.repeat(xs, n_events_per_pixel)
        ys_repeated = np.repeat(ys, n_events_per_pixel)

        return np.column_stack([timestamps, xs_repeated, ys_repeated, polarities])

In [ ]:
sequences = ['canada', 'clamp', 'lighter', 'ruler', 'spinner', 'tag', 'traumel']

In [ ]:
#Make augmentation for 'Augmentation A'

for seq in tqdm(sequences, desc='sequences'):
    base = f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean/{seq}'
    csv_path = os.path.join(base, 'events.csv')
    aps_dir = os.path.join(base, 'aps')
    out_dir = os.path.join(base, 'augmented')
    os.makedirs(out_dir, exist_ok=True)
    augmentor = FlashAugmentor(csv_path, aps_dir)
    log_path = os.path.join(out_dir, 'augmentation_log.txt')
    with open(log_path, 'w') as log:
        for i in tqdm(range(10), desc=seq, leave=False):
            new_events = augmentor.augment()
            out_path = os.path.join(out_dir, f'events_{i+1}.csv')
            pd.DataFrame(new_events, columns=['t', 'x', 'y', 'p']).to_csv(out_path, index=False)
            log.write(f'events_{i+1}.csv — total: {len(new_events)} (+{len(new_events) - len(augmentor.events)} generated)\n')

In [ ]:
#Make augmentation for 'Augmentation B'

for seq in tqdm(sequences, desc='sequences'):
    base = f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean/{seq}'
    csv_path = os.path.join(base, 'events.csv')
    aps_dir = os.path.join(base, 'aps')
    out_dir = os.path.join(base, 'augmented2')
    os.makedirs(out_dir, exist_ok=True)
    print(f'\n[{seq}] loading events...')
    augmentor = FlashAugmentor2(csv_path, aps_dir)
    print(f'[{seq}] loaded {len(augmentor.events)} events, {augmentor.num_frames} frames')
    log_path = os.path.join(out_dir, 'augmentation_log.txt')
    with open(log_path, 'w') as log:
        for i in tqdm(range(5), desc=seq, leave=False):
            print(f'  [{seq}] augmenting {i+1}/5...')
            new_events, flashes = augmentor.augment()
            print(f'  [{seq}] generated {len(new_events)} events ({len(flashes)} flashes), saving...')
            out_path = os.path.join(out_dir, f'events_{i+1}.csv')
            pd.DataFrame(new_events, columns=['t', 'x', 'y', 'p']).to_csv(out_path, index=False)
            print(f'  [{seq}] saved events_{i+1}.csv')
            log.write(f'events_{i+1}.csv — total: {len(new_events)} (+{len(new_events) - len(augmentor.events)} generated)\n')
            log.write(f'Planned {len(flashes)} flashes: {flashes}\n')
    print(f'[{seq}] done')

#Fine-tine with augmented data

In [ ]:
!pip install jpeg4py
!pip install lmdb
!pip install torchinfo
!pip install visdom
!pip install spikingjelly

In [ ]:
%cd /content/

!git clone https://github.com/YmShan/SDTrack.git
!cd SDTrack/SDTrack-Event

In [ ]:
import os
import shutil
import re
from pathlib import Path
from tqdm.notebook import tqdm
import time

import torch

In [ ]:
# Copy yaml and illumination.py

src = '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/SDTrack-tiny-illumination.yaml'
dst = '/content/SDTrack/SDTrack-Event/experiments/SDTrack'
shutil.copy(src, dst)


shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/illumination.py',
    '/content/SDTrack/SDTrack-Event/lib/train/dataset/illumination.py'
)

In [ ]:
#Put weights into correct folders(1х4)

os.makedirs('/content/SDTrack/SDTrack-Event/pretrained_models', exist_ok=True)
os.makedirs('/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-illumination', exist_ok=True)

shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/SDTrack-tiny-1x4.pth',
    '/content/SDTrack/SDTrack-Event/pretrained_models/'
)

shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/SDTrack_ep0100.pth.tar',
    '/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-illumination/'
)


In [ ]:
#Update scripts
shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/sdtrack_update_2.py',
    '/content/SDTrack/SDTrack-Event/sdtrack_update_2.py'
)

!python /content/SDTrack/SDTrack-Event/sdtrack_update_2.py


shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/sd_update_illumination.py',
    '/content/SDTrack/SDTrack-Event/sd_update_illumination.py'
)


!python /content/SDTrack/SDTrack-Event/sd_update_illumination.py

In [1]:
# Copy augmented sequences

base_drive = '/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean'
base_dst = '/content/SDTrack/SDTrack-Event/data/train'
sequences = ['bracket', 'canada', 'clamp', 'lighter', 'ruler', 'spinner', 'tag', 'traumel']

for seq in tqdm(sequences):
    gt = f'{base_drive}/{seq}/groundtruth_rect.txt'
    for i in range(1, 11):
        dst = f'{base_dst}/{seq}_aug{i}'
        os.makedirs(dst, exist_ok=True)
        shutil.copy(gt, f'{dst}/groundtruth_rect.txt')
        shutil.copytree(
            f'{base_drive}/{seq}/augmented/gtp_{i}',
            f'{dst}/inter1_stack_3008'
        )
        print(f'Done: {seq}_aug{i}')

In [ ]:
# Split on test and validation set

BASE = '/content/SDTrack/SDTrack-Event'
train_dir = Path(f'{BASE}/data/train')
data_specs = Path(f'{BASE}/lib/train/data_specs')

all_seqs = sorted([s for s in os.listdir(train_dir) if os.path.isdir(train_dir / s)])
print(f"Found {len(all_seqs)} sequences")

val_seqs = [s for s in all_seqs if s.endswith('_aug1')]
train_seqs = [s for s in all_seqs if s not in val_seqs]

(data_specs / 'illumination_train.txt').write_text('\n'.join(train_seqs) + '\n')
(data_specs / 'illumination_val.txt').write_text('\n'.join(val_seqs) + '\n')

print(f"Train: {len(train_seqs)} sequences")
print(f"Val: {len(val_seqs)} sequences")

In [ ]:
# Reset epoch counter to 0 in the checkpoint to resume training from scratch

import sys
sys.path.insert(0, '/content/SDTrack/SDTrack-Event')

import torch
ckpt_path = '/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-illumination/SDTrack_ep0100.pth.tar'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
ckpt['epoch'] = 0
torch.save(ckpt, ckpt_path)
print('Done, epoch set to 0')

In [ ]:
#Run fine_tune

%cd /content/SDTrack/SDTrack-Event

!python lib/train/run_training.py \
    --script SDTrack \
    --config SDTrack-tiny-illumination \
    --save_dir /content/SDTrack/SDTrack-Event/output \
    --seed 42

In [ ]:
# Copy fine_tuned checkpoints to drive

src = '/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-illumination'
dst = '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/fine_tune_weights'

os.makedirs(dst, exist_ok=True)

for f in os.listdir(src):
    if f != 'SDTrack_ep0100.pth.tar':
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
        print(f'Copied: {f}')

# Re-detection(you can apply follow section to standard checkpoints or fine-tuned)

In [ ]:
!pip install jpeg4py
!pip install lmdb
!pip install torchinfo
!pip install visdom
!pip install spikingjelly

In [ ]:
%cd /content/

!git clone https://github.com/YmShan/SDTrack.git
!cd SDTrack/SDTrack-Event

In [ ]:
import os
import shutil
import re
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np

In [ ]:
#Redetection update

src_running = '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/redetection/running.py'
dst_running = '/content/SDTrack/SDTrack-Event/lib/test/evaluation/running.py'

src_tracking = '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/redetection/tracker.py'
dst_tracking = '/content/SDTrack/SDTrack-Event/lib/test/evaluation/tracker.py'

src_sd = '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/redetection/SDTrack_redetect.py'
dst_sd = '/content/SDTrack/SDTrack-Event/lib/test/tracker/SDTrack.py'

shutil.copy2(src_running, dst_running)
shutil.copy2(src_tracking, dst_tracking)
shutil.copy2(src_sd, dst_sd)

In [ ]:
#For standard checkpoint


%cd /content/SDTrack/SDTrack-Event
!python tracking/test.py SDTrack SDTrack-tiny-fe108 --dataset eotb --threads 4 --num_gpus 1


In [ ]:
#Update scripts
shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/sdtrack_update_2.py',
    '/content/SDTrack/SDTrack-Event/sdtrack_update_2.py'
)


!python /content/SDTrack/SDTrack-Event/sdtrack_update_2.py

In [ ]:
initials = 'seya'
sequences = ['key_1:2hz', 'key_1hz', 'key_2hz', 'lamp_1:2hz', 'lamp_1hz', 'lamp_2hz',
             'repeater_1:2hz', 'repeater_1hz', 'repeater_2hz', 'soap_1:2hz',
             'soap_1hz', 'soap_2hz', 'white_1:2hz', 'white_1hz', 'white_2hz',
             'cleaner_1:2hz', 'cleaner_1hz', 'cleaner_2hz', 'knife_1:2hz', 'knife_1hz', 'knife_2hz',
             'longus_1:2hz', 'longus_1hz', 'longus_2hz', 'nonstop_1:2hz',
             'nonstop_1hz', 'nonstop_2hz', 'rexona_1:2hz', 'rexona_1hz', 'rexona_2hz']

In [ ]:
for seq in tqdm(sequences):

    os.makedirs(f'/content/SDTrack/SDTrack-Event/data/test/{seq}', exist_ok=True)
    os.makedirs('/content/SDTrack/SDTrack-Event/data/train/', exist_ok=True)

    shutil.copy(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/groundtruth_rect.txt',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}'
    )

    shutil.copytree(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/aps',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}/img'
    )

    shutil.copytree(
    f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/inter1_stack_3008',
    f'/content/SDTrack/SDTrack-Event/data/test/{seq}/inter1_stack_3008'
    )


In [ ]:
# Launch SDTracker

!rm -rf '/content/SDTrack/SDTrack-Event/output/test'

%cd /content/SDTrack/SDTrack-Event

!python tracking/test.py SDTrack SDTrack-tiny-fe108 --dataset eotb --threads 1 --num_gpus 1

# Kalman(you can apply follow section to standard checkpoints or fine-tuned)

In [ ]:
!pip install jpeg4py
!pip install lmdb
!pip install torchinfo
!pip install visdom
!pip install spikingjelly

In [ ]:
%cd /content/

!git clone https://github.com/YmShan/SDTrack.git
!cd SDTrack/SDTrack-Event

In [ ]:
import os
import shutil
import re
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np

In [ ]:
src = '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/kalman'
dst = '/content/SDTrack/SDTrack-Event/lib/test'

shutil.copy(f'{src}/SDTrack.py',  f'{dst}/tracker/SDTrack.py')
shutil.copy(f'{src}/running.py',  f'{dst}/evaluation/running.py')
shutil.copy(f'{src}/tracker.py',  f'{dst}/evaluation/tracker.py')
print("Files replaced")

In [ ]:
# Copy checkpoints

os.makedirs('/content/SDTrack/SDTrack-Event/pretrained_models', exist_ok=True)
os.makedirs('/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-fe108', exist_ok=True)

shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/SDTrack-tiny-1x4.pth',
    '/content/SDTrack/SDTrack-Event/pretrained_models/'
)

shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/SDTrack_ep0100.pth.tar',
    '/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-fe108/'
)

In [ ]:
#Update scripts
shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/sdtrack_update_2.py',
    '/content/SDTrack/SDTrack-Event/sdtrack_update_2.py'
)


!python /content/SDTrack/SDTrack-Event/sdtrack_update_2.py

In [ ]:
initials = 'seya'
sequences = ['key_1:2hz', 'key_1hz', 'key_2hz', 'lamp_1:2hz', 'lamp_1hz', 'lamp_2hz',
             'repeater_1:2hz', 'repeater_1hz', 'repeater_2hz', 'soap_1:2hz',
             'soap_1hz', 'soap_2hz', 'white_1:2hz', 'white_1hz', 'white_2hz',
             'cleaner_1:2hz', 'cleaner_1hz', 'cleaner_2hz', 'knife_1:2hz', 'knife_1hz', 'knife_2hz',
             'longus_1:2hz', 'longus_1hz', 'longus_2hz', 'nonstop_1:2hz',
             'nonstop_1hz', 'nonstop_2hz', 'rexona_1:2hz', 'rexona_1hz', 'rexona_2hz']

In [ ]:
#Copy sequences

for seq in tqdm(sequences):

    os.makedirs(f'/content/SDTrack/SDTrack-Event/data/test/{seq}', exist_ok=True)
    os.makedirs('/content/SDTrack/SDTrack-Event/data/train/', exist_ok=True)

    shutil.copy(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/groundtruth_rect.txt',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}'
    )

    shutil.copytree(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/aps',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}/img'
    )

    shutil.copytree(
    f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/inter1_stack_3008',
    f'/content/SDTrack/SDTrack-Event/data/test/{seq}/inter1_stack_3008'
    )


In [ ]:
# Launch SDTracker

!rm -rf '/content/SDTrack/SDTrack-Event/output/test'

%cd /content/SDTrack/SDTrack-Event

!python tracking/test.py SDTrack SDTrack-tiny-fe108 --dataset eotb --threads 1 --num_gpus 1

#Kalman + re-detect(illumination checkpoint)

In [ ]:
!pip install jpeg4py
!pip install lmdb
!pip install torchinfo
!pip install visdom
!pip install spikingjelly

In [ ]:
%cd /content/

!git clone https://github.com/YmShan/SDTrack.git
!cd SDTrack/SDTrack-Event

In [ ]:
src = '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/kalman'
dst = '/content/SDTrack/SDTrack-Event/lib/test'

shutil.copy(f'{src}/SDTrack_kalman_redetect',  f'{dst}/tracker/SDTrack.py')
shutil.copy(f'{src}/running.py',  f'{dst}/evaluation/running.py')
shutil.copy(f'{src}/tracker.py',  f'{dst}/evaluation/tracker.py')
print("Files replaced")

In [ ]:
#Put weights into correct folders

os.makedirs('/content/SDTrack/SDTrack-Event/pretrained_models', exist_ok=True)

shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/SDTrack-tiny-1x4.pth',
    '/content/SDTrack/SDTrack-Event/pretrained_models/'
)

os.makedirs('/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-illumination', exist_ok=True)
shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/fine_tune_weights_3/SDTrack_ep0030.pth.tar',
    '/content/SDTrack/SDTrack-Event/output/checkpoints/train/SDTrack/SDTrack-tiny-illumination/'
)


In [ ]:
# Copy yaml and illumination.py

src = '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/SDTrack-tiny-illumination.yaml'
dst = '/content/SDTrack/SDTrack-Event/experiments/SDTrack'
shutil.copy(src, dst)


shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/illumination.py',
    '/content/SDTrack/SDTrack-Event/lib/train/dataset/illumination.py'
)

In [ ]:
#Update scripts
shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/sdtrack_update_2.py',
    '/content/SDTrack/SDTrack-Event/sdtrack_update_2.py'
)

!python /content/SDTrack/SDTrack-Event/sdtrack_update_2.py


shutil.copy(
    '/content/drive/MyDrive/Diploma_project/SDTrack_pretrained/sd_update_illumination.py',
    '/content/SDTrack/SDTrack-Event/sd_update_illumination.py'
)

!python /content/SDTrack/SDTrack-Event/sd_update_illumination.py

In [ ]:
initials = 'seya'
sequences = ['key_1:2hz', 'key_1hz', 'key_2hz', 'lamp_1:2hz', 'lamp_1hz', 'lamp_2hz',
             'repeater_1:2hz', 'repeater_1hz', 'repeater_2hz', 'soap_1:2hz',
             'soap_1hz', 'soap_2hz', 'white_1:2hz', 'white_1hz', 'white_2hz',
             'cleaner_1:2hz', 'cleaner_1hz', 'cleaner_2hz', 'knife_1:2hz', 'knife_1hz', 'knife_2hz',
             'longus_1:2hz', 'longus_1hz', 'longus_2hz', 'nonstop_1:2hz',
             'nonstop_1hz', 'nonstop_2hz', 'rexona_1:2hz', 'rexona_1hz', 'rexona_2hz']

In [ ]:
#Copy sequences

for seq in tqdm(sequences):

    os.makedirs(f'/content/SDTrack/SDTrack-Event/data/test/{seq}', exist_ok=True)
    os.makedirs('/content/SDTrack/SDTrack-Event/data/train/', exist_ok=True)

    shutil.copy(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/groundtruth_rect.txt',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}'
    )

    shutil.copytree(
        f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/aps',
        f'/content/SDTrack/SDTrack-Event/data/test/{seq}/img'
    )

    shutil.copytree(
    f'/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/{initials}/flicker/{seq}/inter1_stack_3008',
    f'/content/SDTrack/SDTrack-Event/data/test/{seq}/inter1_stack_3008'
    )


In [ ]:
%cd /content/SDTrack/SDTrack-Event

!python tracking/test.py SDTrack SDTrack-tiny-illumination --dataset eotb --threads 2 --num_gpus 1